# Luxembourg LOD2 屋顶指标与检索实验

目标：从同一套 Luxembourg LOD2 建筑派生 G1/M1 与 G2/M2，先在固定抽取的约 500 个街区上完成指标计算和成对检索。

安全边界：已有 `block`、`lod1`、`lod2`、`summary` schema 只读；所有数据库写入仅允许进入 `exp_lod2_roof_lu`。默认不执行人工评分与批量制图。

In [ ]:
from __future__ import annotations

import importlib.metadata as metadata
import json
import os
from datetime import datetime, timezone
from pathlib import Path
import random
import sys

import numpy as np
import pandas as pd

def find_notebook_dir() -> Path:
    current = Path.cwd().resolve()
    if (current / "database.py").is_file():
        return current
    for parent in (current, *current.parents):
        candidate = parent / "casebase" / "lod2_roof_retrieval_experiment"
        if (candidate / "database.py").is_file():
            return candidate
    raise FileNotFoundError("请从项目内运行本 notebook。")

NOTEBOOK_DIR = find_notebook_dir()
os.chdir(NOTEBOOK_DIR)
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import block_metrics as bm
import database as db
import retrieval as rt

RANDOM_SEED = 20260908
EXPERIMENT_SCHEMA = "exp_lod2_roof_lu"
TARGET_BLOCK_COUNT = 500  # 改为 None 可处理全部匹配街区
COVERAGE_THRESHOLD = 0.80
CORRELATION_THRESHOLD = 0.90
POOL_SIZE = 50
TOP_K = 5
LAMBDA_VALUES = [0.20, 0.35, 0.50]
WRITE_EXPERIMENT_SCHEMA = True
ENV_PATH = None  # 若当前工作树没有 .env，在这里填入其绝对路径

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## 实验顺序

1. 读取源数据并固定街区样本；2. 计算建筑与街区指标；3. 记录宽松质检结果；4. 固定 M1 特征；5. 在共享候选池中比较 M1 与 M2。

In [ ]:
connection = db.connect_from_env(ENV_PATH)
metric_srid = db.fetch_lod2_metric_srid(connection)
all_blocks = db.fetch_luxembourg_blocks(connection, metric_srid)
if TARGET_BLOCK_COUNT is None or TARGET_BLOCK_COUNT >= len(all_blocks):
    sampled_blocks = all_blocks.copy()
else:
    sampled_blocks = all_blocks.sample(
        n=TARGET_BLOCK_COUNT, random_state=RANDOM_SEED
    ).sort_values("block_id").reset_index(drop=True)
sampled_block_ids = sampled_blocks["block_id"].astype(str).tolist()
pd.Series({
    "metric_srid": metric_srid,
    "available_blocks": len(all_blocks),
    "sampled_blocks": len(sampled_blocks),
})

## 写入范围确认

下面先显示并校验三个唯一允许的写入目标。关闭 `WRITE_EXPERIMENT_SCHEMA` 时，文件输出仍会生成，但数据库不会新建或替换实验表。

In [ ]:
assert EXPERIMENT_SCHEMA == db.EXPERIMENT_SCHEMA
WRITE_TARGETS = [db.BUILDING_TABLE, db.BLOCK_TABLE, db.RETRIEVAL_TABLE]
for target in WRITE_TARGETS:
    db.assert_experiment_relation(target)
display(pd.DataFrame({"allowed_write_target": WRITE_TARGETS}))
if WRITE_EXPERIMENT_SCHEMA:
    db.ensure_experiment_schema(connection)

## 建筑层指标

G1 高度使用屋顶水平投影面积加权平均高程减地面高程；屋顶坡度使用真实三维屋面面积加权。多 surface 被正常合并，只有核心指标无法计算时才排除建筑。

In [ ]:
building_records = db.iter_luxembourg_buildings(
    connection, metric_srid, block_ids=sampled_block_ids
)
building_metrics = bm.compute_building_metrics(building_records)
pd.Series({
    "buildings": len(building_metrics),
    "usable_buildings": int(building_metrics["is_usable"].sum()),
    "usable_share": float(building_metrics["is_usable"].mean()),
})

In [ ]:
building_quality = (
    building_metrics.assign(level="building", object_id=building_metrics["building_id"])
    [["level", "object_id", "is_usable", "quality_codes"]]
)
building_quality["quality_codes"].value_counts(dropna=False).head(15)

## 街区层 M1/M2 指标

M1 复用既有 LOD1 表征的定义思路，但所有数值都由本次同源 LOD2 样本重新计算。M2 完整保留最终 M1，并增加平均坡度、屋顶相对起伏中位数和建筑间坡度离散度。

In [ ]:
block_metrics = bm.compute_block_metrics(
    sampled_blocks, building_metrics, coverage_threshold=COVERAGE_THRESHOLD
)
block_quality = (
    block_metrics.assign(level="block", object_id=block_metrics["block_id"])
    [["level", "object_id", "is_usable", "quality_codes"]]
)
quality_flags = pd.concat([building_quality, block_quality], ignore_index=True)
pd.Series({
    "blocks": len(block_metrics),
    "usable_blocks": int(block_metrics["is_usable"].sum()),
    "usable_share": float(block_metrics["is_usable"].mean()),
})

In [ ]:
usable_blocks = block_metrics.loc[block_metrics["is_usable"]].copy()
if len(usable_blocks) <= POOL_SIZE:
    raise RuntimeError(
        f"仅有 {len(usable_blocks)} 个可用街区，不足以建立 {POOL_SIZE} 个候选的检索池。"
    )
final_m1_features, correlation_decisions = bm.select_final_m1_features(
    usable_blocks, threshold=CORRELATION_THRESHOLD
)
display(pd.Series({"M1候选数": len(bm.M1_CANDIDATE_FEATURES), "最终M1数": len(final_m1_features)}))
display(correlation_decisions)
final_m1_features

## 保存指标与质检记录

输出目录中的 CSV、Parquet 和 JSON 是可复核的运行记录，不提交到 Git。

In [ ]:
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
quality_flags.to_csv(OUTPUT_DIR / "quality_flags.csv", index=False, encoding="utf-8-sig")
building_metrics.to_parquet(OUTPUT_DIR / "building_metrics.parquet", index=False)
block_metrics.to_parquet(OUTPUT_DIR / "block_metrics.parquet", index=False)
if WRITE_EXPERIMENT_SCHEMA:
    db.replace_building_metrics(connection, building_metrics)
    db.replace_block_metrics(connection, block_metrics)

## 固定开发查询与成对检索

两种方法共享按 M1 总体形态距离选出的 50 个候选；M1 直接排序，M2 只在这 50 个候选内加入屋顶距离重排。这样屋顶信息不会把总体形态完全不可比的案例拉入结果。

In [ ]:
development_queries, retrieval_results, general_scaler, roof_scaler = (
    rt.run_development_retrieval(
        usable_blocks,
        general_features=final_m1_features,
        roof_features=bm.ROOF_FEATURES,
        pool_size=POOL_SIZE,
        top_k=TOP_K,
        lambda_values=LAMBDA_VALUES,
    )
)
display(development_queries)
display(retrieval_results.head(20))

In [ ]:
development_queries.to_csv(
    OUTPUT_DIR / "development_queries.csv", index=False, encoding="utf-8-sig"
)
retrieval_results.to_csv(
    OUTPUT_DIR / "development_retrieval_results.csv", index=False, encoding="utf-8-sig"
)
if WRITE_EXPERIMENT_SCHEMA:
    db.replace_retrieval_results(connection, retrieval_results)

experiment_config = {
    "execution_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "random_seed": RANDOM_SEED,
    "experiment_schema": EXPERIMENT_SCHEMA,
    "metric_srid": metric_srid,
    "target_block_count": TARGET_BLOCK_COUNT,
    "sampled_block_count": len(sampled_blocks),
    "usable_block_count": len(usable_blocks),
    "coverage_threshold": COVERAGE_THRESHOLD,
    "correlation_threshold": CORRELATION_THRESHOLD,
    "m1_features_before_pruning": bm.M1_CANDIDATE_FEATURES,
    "m1_features_after_pruning": final_m1_features,
    "roof_features": bm.ROOF_FEATURES,
    "correlation_decisions": correlation_decisions.to_dict(orient="records"),
    "general_scaler_medians": general_scaler.medians.to_dict(),
    "general_scaler_iqrs": general_scaler.iqrs.to_dict(),
    "roof_scaler_medians": roof_scaler.medians.to_dict(),
    "roof_scaler_iqrs": roof_scaler.iqrs.to_dict(),
    "pool_size": POOL_SIZE,
    "top_k": TOP_K,
    "lambda_values": LAMBDA_VALUES,
    "package_versions": {name: metadata.version(name) for name in [
        "numpy", "pandas", "scipy", "scikit-learn", "shapely", "psycopg2", "pyarrow"
    ]},
}
with (OUTPUT_DIR / "experiment_config.json").open("w", encoding="utf-8") as file:
    json.dump(experiment_config, file, ensure_ascii=False, indent=2)
connection.close()

## 初步结果检查

这里先检查屋顶信息造成了多少 Top-5 替换，不把排名变化直接解释成检索质量改善。后续评价方案确定后，再将这些候选统一展示并评价。

In [ ]:
m1_sets = (
    retrieval_results.query("method == 'M1'")
    .groupby("query_id")["candidate_id"].apply(set)
)
replacement_rows = []
for (query_id, lambda_value), group in retrieval_results.query("method == 'M2'").groupby(
    ["query_id", "lambda_value"]
):
    overlap = len(m1_sets[query_id] & set(group["candidate_id"]))
    replacement_rows.append({
        "query_id": query_id,
        "lambda_value": lambda_value,
        "top5_replaced": TOP_K - overlap,
    })
pd.DataFrame(replacement_rows)

## 后续

- 先查看质检比例、指标分布和 5 个开发查询是否合理。
- 如可用街区明显少于预期，优先检查原因代码，而不是立即提高质检复杂度。
- 确认检索结果后，再讨论不依赖大规模专家评分的评价与案例解释方案。